# `ssa_friction_inv_schoof`

> Generated from `config/ais_0.1.py` by `config/make_step_notebooks.py`. Edit the driver and regenerate; changes made here are overwritten.

Schoof friction inversion for the HO track, started from the Budd p=q=1 field. Submits the solve; run again once it has finished to save the result.

**Reads** (`load_model` calls in this step):
- `f'{model_dir}/AIS3_param.nc'`
- `f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/{schoof_budd_run}/{schoof_budd_run}.nc'`

**Writes** (`save_model` calls in this step):
- `_out = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/{schoof_run}/{schoof_run}.nc'`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_ssa_friction_inv_schoof.pbs`):

```bash
qsub -q normal -l ncpus=4,mem=96GB,walltime=01:00:00 -v NB=ais_0.1/10_ssa_friction_inv_schoof.ipynb launch_notebook.pbs
```

The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: none.

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

## Setup

The preamble of `ais_0.1.py`: imports, helpers, settings and the data catalog.

#### Imports

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os
import glob
import copy
import sys
from pathlib import Path

# Site settings (paths, NCI project, ISSM build): edit config/site.env, not this file. Found
# from this script's folder, or from the working directory upwards in a generated notebook.
_here = Path(globals().get('__file__', Path.cwd() / '_')).resolve().parent
sys.path[:0] = [str(p) for p in (_here, *_here.parents) if (p / 'ais3_site.py').exists()][:1]
import ais3_site as site

#### `racmo_mean_smb()` — Time-mean RACMO2.4p1 smbgl over calendar years y0..y1 on md's mesh, in m ice eq/yr

In [ ]:
def racmo_mean_smb(md, y0, y1):
    """Time-mean RACMO2.4p1 smbgl over calendar years y0..y1 on md's mesh, in m ice eq/yr.

    Same unit handling as historical_dhdt_tuning (monthly kg/m^2 summed, divided by rho_ice),
    averaged over the years. Uses the module-level `catalog`.
    """
    data = catalog.load_dataset('racmo2.4p1_monthly_11km_1979-2023')
    smbgl = data['smbgl']
    rx, ry = pyissm.tools.general.ll_to_xy(data['lat'].values, data['lon'].values, -1)
    sel = smbgl.sel(time=(smbgl['time.year'] >= y0) & (smbgl['time.year'] <= y1))
    n_years = y1 - y0 + 1
    assert sel.sizes['time'] == 12 * n_years, f"expected {12 * n_years} months, got {sel.sizes['time']}"
    mean_myr = (sel.sum('time') / n_years / md.materials.rho_ice).squeeze('height').to_numpy()
    out = pyissm.data.interp.points_to_mesh(rx, ry, mean_myr, md.mesh.x, md.mesh.y)
    n_nan = int(np.isnan(out).sum())
    print(f"   RACMO {y0}-{y1} mean SMB: mesh-mean {np.nanmean(out):.3f} m ice eq/yr, "
          f"{n_nan} vertices outside RACMO coverage set to 0")
    return np.nan_to_num(out, nan=0.0)

#### `sync_mesh_z()` — Recompute the extruded mesh's vertex z from the current geometry (base + sigma*H)

In [ ]:
def sync_mesh_z(md):
    """Recompute the extruded mesh's vertex z from the current geometry (base + sigma*H).

    Keeps each column's relative layer spacing (sigma from the existing z), so it works even if
    md.mesh.z is stale -- e.g. after geometry was edited without touching z, as the thickness
    restoration (restore_floored_thickness.py) and the historical chunk resume did.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    n_layers = nv // nvb
    assert n_layers * nvb == nv
    assert np.asarray(md.mesh.vertexonbase).astype(bool).ravel()[:nvb].all()
    assert np.asarray(md.mesh.vertexonsurface).astype(bool).ravel()[-nvb:].all()
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(n_layers, nvb)
    span = z[-1] - z[0]
    safe = np.abs(span) > 1e-9
    sigma = np.where(safe[None, :], (z - z[0]) / np.where(safe, span, 1.0)[None, :],
                     np.linspace(0.0, 1.0, n_layers)[:, None])
    base = np.asarray(md.geometry.base, dtype=float).ravel()[:nvb]
    thick = np.asarray(md.geometry.thickness, dtype=float).ravel()[:nvb]
    d_base = np.abs(z[0] - base)
    d_surf = np.abs(z[-1] - (base + thick))
    print(f"   mesh.z vs geometry before sync: base max {d_base.max():.3f} m ({int((d_base > 0.01).sum())} "
          f"columns off by >1 cm), surface max {d_surf.max():.3f} m ({int((d_surf > 0.01).sum())} columns)")
    md.mesh.z = (base[None, :] + sigma * thick[None, :]).ravel()

#### `use_ssa_budd_friction_no_pins()` — Swap the HO model's Schoof friction and velocity pins for the SSA track's friction

In [ ]:
def use_ssa_budd_friction_no_pins(md):
    """Swap the HO model's Schoof friction and velocity pins for the SSA track's friction.

    The HO Schoof field cannot run without its ~176k pinned columns (spcvx/spcvy = observed
    velocity): released on the 1995 state it reaches 2e8 m/yr, and the pinned columns themselves
    thicken by up to ~3 km over the historical run (worklog 6.21, 6.23). The SSA track's Budd p=q=1
    field (coefficient and Ehrenfeucht N, coupling 3, limit 0.07) runs pin-free on the same 1995
    state, max 5.5 km/yr (worklog 6.27). Copied up every column from models/AIS3_proj_ssa_init.nc,
    whose 2D mesh is the HO base layer (checked).
    """
    import netCDF4
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    nl = nv // nvb
    s = netCDF4.Dataset(f'{model_dir}/AIS3_proj_ssa_init.nc')
    assert np.allclose(np.asarray(s['mesh'].variables['x'][:]), np.asarray(md.mesh.x)[:nvb])
    assert np.allclose(np.asarray(s['mesh'].variables['y'][:]), np.asarray(md.mesh.y)[:nvb])
    sf = s['friction']
    assert sf.getncattr('classtype') == 'friction.default' and int(sf.getncattr('coupling')) == 3
    assert np.all(np.asarray(sf.variables['p'][:]) == 1) and np.all(np.asarray(sf.variables['q'][:]) == 1)
    C2 = np.asarray(sf.variables['coefficient'][:], dtype=float).ravel()
    N2 = np.asarray(sf.variables['effective_pressure'][:], dtype=float).ravel()
    lim = float(sf.getncattr('effective_pressure_limit'))
    s.close()
    print(f"-- Friction: {type(md.friction).__name__} -> SSA Budd p=q=1 (coupling 3, limit {lim}); "
          f"grounded C median {np.median(C2[(np.asarray(md.mask.ocean_levelset).ravel()[:nvb] > 0)]):.0f}")
    md.friction = pyissm.model.classes.friction.default()
    md.friction.coefficient = np.tile(C2, nl)
    md.friction.p = np.ones(md.mesh.numberofelements)
    md.friction.q = np.ones(md.mesh.numberofelements)
    md.friction.coupling = 3
    md.friction.linearize = 0
    md.friction.effective_pressure = np.tile(N2, nl)
    md.friction.effective_pressure_limit = lim
    pinned = np.isfinite(np.asarray(md.stressbalance.spcvx, dtype=float).ravel())
    print(f"-- Releasing all velocity pins: {int(pinned.sum())} vertices "
          f"({int(pinned[:nvb].sum())} columns)")
    md.stressbalance.spcvx = np.full(nv, np.nan)
    md.stressbalance.spcvy = np.full(nv, np.nan)
    md.stressbalance.spcvz = np.full(nv, np.nan)

#### `adopt_final_state()` — Make the transient's last step the model's own state before save_model

In [ ]:
def adopt_final_state(md):
    """Make the transient's last step the model's own state before save_model.

    solve() fills md.results only, so without this save_model writes the transient's INPUT
    geometry -- the reason neither relaxation ever reached downstream steps (worklog 4y).
    Carries Thickness and MaskOceanLevelset; base/surface are recomputed from them with the
    hydrostatic-floating/bed-grounded formula (raw output Base can sit below bed at newly
    grounded vertices, worklog 4o); velocities go to md.initialization when saved.
    """
    ts = md.results.TransientSolution
    nv = md.mesh.numberofvertices

    def last(field):
        if isinstance(ts, list):
            return np.asarray(getattr(ts[-1], field), dtype=float).ravel().copy()
        raw = np.asarray(getattr(ts, field), dtype=float).ravel()
        return raw.reshape(raw.size // nv, nv)[-1].copy()

    thick = last('Thickness')
    ol = last('MaskOceanLevelset')
    bed = np.asarray(md.geometry.bed, dtype=float).ravel()
    ri = float(md.materials.rho_ice)
    rw = float(md.materials.rho_water)
    base = np.maximum(np.where(ol >= 0, bed, -thick * ri / rw), bed)
    old_thick = np.asarray(md.geometry.thickness, dtype=float).ravel()
    old_ol = np.asarray(md.mask.ocean_levelset, dtype=float).ravel()
    md.geometry.thickness = thick
    md.geometry.base = base
    md.geometry.surface = base + thick
    md.mask.ocean_levelset = ol
    set_vel = []
    for field in ('Vx', 'Vy', 'Vz', 'Vel'):
        try:
            setattr(md.initialization, field.lower(), last(field))
            set_vel.append(field)
        except (AttributeError, ValueError):
            pass
    d = thick - old_thick
    print(f"   adopted final transient state: mean|dH| {np.abs(d).mean():.3f} m, max|dH| "
          f"{np.abs(d).max():.2f} m; grounded/floating changed at "
          f"{100 * np.mean((ol >= 0) != (old_ol >= 0)):.3f}% of vertices; initialization set: {set_vel}")

#### `rescale_rheology_B_d2()` — Rescale md's 3D rheology_B column by column to a 2D reference B, keeping the vertical

In [ ]:
def rescale_rheology_B_d2(md, B_ref_2d):
    """Rescale md's 3D rheology_B column by column to a 2D reference B, keeping the vertical
    profile ("D2", worklog 5c).

    Grounded columns match the deformation-weighted column B, B_eff = (sum w B^-n / sum w)^(-1/n)
    with w = (1 - sigma)^(n+1) (sigma 0 at the bed), so the soft basal layers count as they do for
    shear flow; floating columns match the plain column mean (plug flow). HO's steady-state
    temperature makes its basal ice much softer than the reference, which made HO flow ~9% too
    fast; with this rescale a diagnostic solve gave grounded surface speed / observed median 1.00.
    Idempotent: a column already matching the reference gets scale 1.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    nl = nv // nvb
    B_ref_2d = np.asarray(B_ref_2d, dtype=float).ravel()
    assert B_ref_2d.size == nvb, f'reference B has {B_ref_2d.size} values, base layer has {nvb}'
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(nl, nvb)
    span = z[-1] - z[0]
    sig = (z - z[0]) / np.where(np.abs(span) > 1e-9, span, 1.0)[None, :]
    w = np.diff(sig, axis=0)
    wsum = w.sum(axis=0)
    Bh = np.asarray(md.materials.rheology_B, dtype=float).ravel().reshape(nl, nvb)
    n = float(np.unique(np.asarray(md.materials.rheology_n).ravel())[0])
    B_mean = (0.5 * (Bh[1:] + Bh[:-1]) * w).sum(axis=0) / np.maximum(wsum, 1e-12)
    wdef = (1.0 - sig) ** (n + 1.0)
    wd = 0.5 * (wdef[1:] + wdef[:-1]) * w
    A_mid = 0.5 * (Bh[1:] ** -n + Bh[:-1] ** -n)
    B_eff = ((wd * A_mid).sum(axis=0) / np.maximum(wd.sum(axis=0), 1e-30)) ** (-1.0 / n)
    grounded = np.asarray(md.mask.ocean_levelset).ravel()[:nvb] > 0
    scale = np.where(grounded, B_ref_2d / B_eff, B_ref_2d / B_mean)
    scale = np.where(wsum > 0, scale, 1.0)
    md.materials.rheology_B = (Bh * scale[None, :]).ravel()
    print(f"   D2 rheology rescale: grounded median {np.median(scale[grounded]):.3f} "
          f"(p10 {np.percentile(scale[grounded], 10):.3f}, p90 {np.percentile(scale[grounded], 90):.3f}), "
          f"floating median {np.median(scale[~grounded]):.3f}")

#### `friction_law_info()` — Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law

In [ ]:
def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)

#### `extract_friction_inversion_domain()` — Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition

In [ ]:
def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds

#### `exp_contains()` — True for points (x, y) inside any polygon of an ISSM .exp file (pure Python, no wrappers)

In [ ]:
def exp_contains(exp_file, x, y):
    """True for points (x, y) inside any polygon of an ISSM .exp file (pure Python, no wrappers).

    Gives the same vertex sets as ContourToNodes(..., 2) for constrain_Schoof.exp and
    PIG_Outline.exp on this mesh (checked 2026-11-17).
    """
    from matplotlib.path import Path as _P
    lines = open(exp_file).read().splitlines()
    inside = np.zeros(np.size(x), dtype = bool)
    pts = np.column_stack([np.ravel(x), np.ravel(y)])
    i = 0
    while i < len(lines):
        if lines[i].startswith('## Name'):
            n = int(lines[i + 3].split()[0])
            poly = np.array([[float(v) for v in lines[i + 5 + k].split()[:2]] for k in range(n)])
            inside |= _P(poly).contains_points(pts)
            i += 5 + n
        else:
            i += 1
    return inside

#### `load_shelf_rheology_B()` — Floating-ice rheology B from the shelf rheology inversion: (extracted vertex ids, Bbar)

In [ ]:
def load_shelf_rheology_B():
    """Floating-ice rheology B from the shelf rheology inversion: (extracted vertex ids, Bbar).

    Reads models/AIS3_ssa_rheology_floating_inv_lcurve/<rheology_lcurve_run>/, the output of the
    ssa_rheology_floating_inv_lcurve step at the chosen point (cf101=1, cf103=10, cf502=1e-17;
    validated grounded RMSE ~60 for the friction inversions warm-started with it). Until
    2026-11-16 this loaded the one-off execution_newB_rheology/run_001_1_10_1e-17 run directly;
    that run is what today's step produces (identical inputs), and finalize_newB_rheology.py
    saved it under run_004_1_10_1e-17. The June 2026 run_004 (before the geometry fixes, RMSE ~114
    when used as a warm start) is kept as run_004_1_10_1e-17.jun30_before_geometry_fixes;
    AIS3_inverted.nc and the existing HO track were built with that June field.
    """
    mr = pyissm.model.io.load_model(
        f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/{rheology_lcurve_run}/{rheology_lcurve_run}.nc')
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## ------------------------------------
## Configure options
## ------------------------------------

#### Settings

In [ ]:
# Change directory to gdata to prevent storage limits in $HOME
os.chdir(site.ROOT)
os.environ['ISSM_DIR'] = site.ISSM_DIR

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = f'{site.ROOT}/execution'

# Define location to save final models
model_dir = f'{site.ROOT}/models'

# Define domain_file
domain_file = f'{site.ROOT}/assets/ais_domain.exp'

# Define param_file
param_file = f'{site.CONFIG}/ais_0.1_param.py'

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = site.STORAGE
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = site.USER
cluster.project = site.PROJECT

# List all steps for clarity
all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_friction_inv_schoof',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
    'projection_ssp',
]

# Define steps to run
# steps = ['process_domain', 'mesh', 'param']
# steps = ['mesh', 'param']
# steps = ['ssa_rheology_floating_inv_sensit']
# steps = ['ssa_rheology_floating_inv_lcurve']
# steps = ['ssa_friction_forward_check']
# steps = ['ssa_friction_forward_check_budd']
# steps = ['param']
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
# steps = ['param']
# steps = ['param']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
steps = ['ho_friction_inv']
# Per-job override via `qsub -v AIS3_STEPS=a,b` -- lets a launcher pick its step without editing
# the shared `steps` above and holding that edit through a multi-day hugemem queue wait.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')
# Suffix for historical_dhdt_tuning's checkpoint file and execution name, so a rerun
# (e.g. AIS3_HIST_TAG=_chunk1rerun) cannot overwrite the finished AIS3_historical_1995_2019.nc.
HIST_TAG = os.environ.get('AIS3_HIST_TAG', '')
# Friction for a FRESH historical start (ignored on resume: the checkpoint carries it).
#   ''            the HO Schoof field and its ~176k pinned columns, as in AIS3_ho_relaxed.nc
#   'budd_nopins' the SSA track's Budd p=q=1 field and no pins (use_ssa_budd_friction_no_pins;
#                 worklog 6.27). Needs its own AIS3_HIST_TAG.
HIST_FRICTION = os.environ.get('AIS3_HIST_FRICTION', '')
assert HIST_FRICTION in ('', 'budd_nopins'), HIST_FRICTION
assert not HIST_FRICTION or HIST_TAG not in ('', '_relaxfix', '_chunk1rerun'), \
    'AIS3_HIST_FRICTION needs a new AIS3_HIST_TAG'
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_friction_inv_sensit']

#### Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = f'{site.ROOT}/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

# Schoof friction field for the HO track, built from scratch by ssa_friction_inv_schoof (see the
# step for the method). schoof_budd_run is the Budd p=q=1 field it starts from: the
# ssa_friction_inv_reg_lcurve output, the same field the SSA track uses. schoof_run is the name
# the result is saved under in models/AIS3_ssa_friction_inv_reg_lcurve/; point
# friction_lcurve_run at it (above) to use it downstream.
schoof_budd_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'
schoof_run = 'schoof_from_budd_cmax2.0'

#### Initialise Data Catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## Step `ssa_friction_inv_schoof`

In [ ]:
steps = ['ssa_friction_inv_schoof']  # this notebook's step; overrides the driver default and AIS3_STEPS

**Section comment from the driver** (kept as written; it records history and may be out of date):

### Schoof friction inversion for the HO track, started from the Budd p=q=1 field

Rebuilds the Schoof (regularised Coulomb) friction field the HO track starts from. The field in production (schoof_m1qn3_tightrestol_cmax2.0, 2026-08-16) was made by a script that was not kept; its inputs were recovered from the run's own .bin (worklog 2026-11-17) and are reproduced here: - geometry, N and the subdomain as in ssa_friction_inv_reg_lcurve (100 m floor keeping the observed surface, N floored at 0.07 of overburden); - ice-shelf B left at AIS3_param.nc's value: the production run did not load an inverted shelf B (its floating B matches AIS3_param.nc exactly, neither shelf inversion); - velocity pinned to observations inside assets/constrain_Schoof.exp; - starting C: the Schoof C that gives the same basal drag, at the observed speed, as the Budd p=q=1 field (tau = C_b^2 N u), with Cmax = 2.0 and m = 1/3. The production start followed this exactly on 97.9% of grounded vertices; on the other 2.1% its drag was lowered by an unrecorded rule (median x0.8), and on 248 of those the Budd drag exceeds the Schoof cap Cmax*N, where no Schoof C matches. Here the drag is capped at schoof_start_cap of Cmax*N instead; - bounds 50..8000 on grounded ice, 0 on floating ice; - cost functions 101 (10, 15 inside assets/PIG_Outline.exp), 103 (23) and 501 (7e-8) on every vertex, no masks; - m1qn3: maxiter 170, maxsteps 800, dxmin 1e-5, dfmin_frac 0.1, gttol 1e-4; stress balance restol 1e-3, reltol 1e-2, maxiter 300. The production run took 36.6 h on 48 cores. This step submits the solve and returns; run the step again once it has finished to load the result and save it to models/AIS3_ssa_friction_inv_reg_lcurve/<schoof_run>/<schoof_run>.nc.

In [ ]:
print("-------------------------------------------------------------")
print(f" SSA SCHOOF FRICTION INVERSION (HO track), from Budd p=q=1")
print("-------------------------------------------------------------")

_out = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/{schoof_run}/{schoof_run}.nc'
_exe = f'{site.ROOT}/execution_ssa_friction_inv_schoof'
if os.path.exists(_out):
    raise SystemExit(f'{_out} exists -- move it aside with mv -n to rebuild it')

#### Loading parameterized model

In [ ]:
print(f"-- Loading parameterized model...")
md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
if type(md.friction).__name__ != 'schoof':
    raise ValueError("ssa_friction_inv_schoof needs AIS3_param.nc built with friction_law='schoof'")

#### Keeping AIS3_param.nc's ice-shelf B (as the production Schoof run did)

In [ ]:
print(f"-- Keeping AIS3_param.nc's ice-shelf B (as the production Schoof run did)...")

#### Removing icebergs from ice levelset

In [ ]:
print('-- Removing icebergs from ice levelset...')
md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

#### Flooring thin ice at 100m (numerical stability), preserving observed surface

In [ ]:
print(f"-- Flooring thin ice at 100m (numerical stability), preserving observed surface...")
_ri = md.materials.rho_ice; _rw = md.materials.rho_water
_H = np.asarray(md.geometry.thickness).ravel().copy()
_ol = np.asarray(md.mask.ocean_levelset).ravel()
_surf0 = np.asarray(md.geometry.surface).ravel().copy()
_H = np.maximum(_H, 100.0)
_flt = _ol < 0
_base = np.empty_like(_H); _surf = np.empty_like(_H)
_base[_flt] = -_H[_flt] * _ri / _rw; _surf[_flt] = _H[_flt] * (1.0 - _ri / _rw)
_surf[~_flt] = _surf0[~_flt]
_base[~_flt] = _surf0[~_flt] - _H[~_flt]
md.geometry.thickness = _H; md.geometry.base = _base; md.geometry.surface = _surf

_lim = 0.07
N = md.friction.effective_pressure.copy()
N[N < 0] = 0
N = np.maximum(N, _lim * md.materials.rho_ice * md.constants.g * _H)
md.friction.effective_pressure = N
md.friction.effective_pressure_limit = _lim

md.inversion.iscontrol = 1
md.verbose.solution = 0
md.verbose.qmu = 0
md.verbose.control = 1

#### Extracting friction-inversion domain (floating/grounded ice-front BC)

In [ ]:
print(f"-- Extracting friction-inversion domain (floating/grounded ice-front BC)...")
mds = extract_friction_inversion_domain(md)
mds.friction.coupling = friction_coupling
_Hs = np.asarray(mds.geometry.thickness).ravel()
_Ns = mds.friction.effective_pressure.copy(); _Ns[_Ns < 0] = 0
mds.friction.effective_pressure = np.maximum(_Ns, 0.07 * mds.materials.rho_ice * mds.constants.g * _Hs)
mds.friction.effective_pressure_limit = 0.07

_x = np.asarray(mds.mesh.x).ravel(); _y = np.asarray(mds.mesh.y).ravel()
_cpos = exp_contains(f'{site.ROOT}/assets/constrain_Schoof.exp', _x, _y)
mds.stressbalance.spcvx[_cpos] = np.asarray(mds.inversion.vx_obs).ravel()[_cpos]
mds.stressbalance.spcvy[_cpos] = np.asarray(mds.inversion.vy_obs).ravel()[_cpos]
mds.stressbalance.spcvz[_cpos] = 0
print(f"   pinned {int(_cpos.sum())} vertices inside constrain_Schoof.exp to observed velocity")

#### Starting C from the Budd p=q=1 field `({schoof_budd_run})`

In [ ]:
print(f"-- Starting C from the Budd p=q=1 field ({schoof_budd_run})...")
mdb = pyissm.model.io.load_model(
    f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/{schoof_budd_run}/{schoof_budd_run}.nc')
_full = np.zeros(md.mesh.numberofvertices)
_full[np.asarray(mdb.mesh.extractedvertices).ravel().astype(int) - 1] = \
    np.asarray(mdb.results.StressbalanceSolution.FrictionCoefficient).ravel()
Cb = _full[np.asarray(mds.mesh.extractedvertices).ravel().astype(int) - 1]
_Cmax = 2.0; _m = 1.0 / 3.0
_N = np.asarray(mds.friction.effective_pressure).ravel()
_u = np.asarray(mds.inversion.vel_obs).ravel() / md.constants.yts   # m/s, as ISSM evaluates the laws
_r = np.minimum(Cb**2 * _N * _u / (_Cmax * _N), schoof_start_cap)   # Budd drag / Schoof cap
with np.errstate(divide = 'ignore', invalid = 'ignore'):
    _C = np.sqrt(_r * _Cmax * _N / (_u**_m * (1.0 - _r**(1.0 / _m))**_m))
_flt_s = np.asarray(mds.mask.ocean_levelset).ravel() < 0
_C[_flt_s | ~np.isfinite(_C)] = 0.05
print(f"   {int(np.sum(~_flt_s & (Cb**2 * _u >= schoof_start_cap * _Cmax)))} grounded vertices capped at "
      f"{schoof_start_cap} of Cmax*N; C {np.min(_C[~_flt_s]):.3g}..{np.max(_C[~_flt_s]):.3g}")
mds.friction.C = _C
mds.friction.Cmax = np.full(mds.mesh.numberofvertices, _Cmax)
mds.friction.m = np.full(np.asarray(mds.friction.m).shape, _m)

#### Defining inversion parameters

In [ ]:
print(f"-- Defining inversion parameters...")
mds.inversion = pyissm.model.classes.inversion.m1qn3(mds.inversion)
mds.inversion.control_parameters = ['FrictionC']
mds.inversion.min_parameters = np.where(_flt_s, 0.0, 50.0)
mds.inversion.max_parameters = np.where(_flt_s, 0.0, 8000.0)
mds.inversion.maxiter = 170
mds.inversion.maxsteps = 800
mds.inversion.dxmin = 1e-5
mds.inversion.dfmin_frac = 0.1
mds.inversion.gttol = 1e-4
_pig = exp_contains(f'{site.ROOT}/assets/PIG_Outline.exp', _x, _y)
mds.inversion.cost_functions = [101, 103, 501]
mds.inversion.cost_functions_coefficients = np.column_stack(
    [np.where(_pig, 15.0, 10.0), np.full(_pig.size, 23.0), np.full(_pig.size, 7e-8)])

mds.transient = pyissm.model.classes.transient.deactivate_all(mds.transient)
mds.stressbalance.restol = 0.001
mds.stressbalance.reltol = 0.01
mds.stressbalance.maxiter = 300
mds.stressbalance.abstol = np.nan
mds.settings.solver_residue_threshold = 1e-3

mds.cluster = copy.deepcopy(cluster)
mds.cluster.executionpath = _exe
mds.settings.waitonlock = 0
mds.miscellaneous.name = schoof_run

_done = os.path.exists(f'{_exe}/{schoof_run}/{schoof_run}.outlog') and \
    'Total elapsed time' in open(f'{_exe}/{schoof_run}/{schoof_run}.outlog').read()
if save and _done:
    print(f"-- Loading the finished solve from {_exe}/{schoof_run}...")
    mds = pyissm.model.execute.solve(mds, 'Stressbalance', runtime_name = False, load_only = True)
    _v = np.asarray(mds.results.StressbalanceSolution.Vel).ravel()
    _vo = np.asarray(mds.inversion.vel_obs).ravel()
    _g = ~_flt_s & (_vo > 0)
    print(f"   grounded RMSE (subdomain): {np.sqrt(np.mean((_v[_g] - _vo[_g])**2)):.2f} m/yr "
          f"(production run: 94.44 on its subdomain, 70.43 after ssa_inverted_solve)")
    os.makedirs(os.path.dirname(_out), exist_ok = True)
    pyissm.model.io.save_model(mds, _out)
    print(f"-- Saved {_out}")
elif os.path.exists(f'{_exe}/{schoof_run}'):
    raise SystemExit(f'{_exe}/{schoof_run} exists but has not finished -- wait for it, or move it aside with mv -n')
else:
    print(f"-- Submitting the Schoof inversion ({cluster.np} cores, up to {cluster.time / 60:.0f} h); "
          f"run this step again when it has finished")
    os.makedirs(_exe, exist_ok = True)
    pyissm.model.execute.solve(mds, 'Stressbalance', runtime_name = False)